# CS-4063 NLP Assignment 2 - i23-2538

GitHub URL: https://github.com/YOUR_USERNAME/i23-2538-NLP-Assignment

This notebook compiles all required outputs for submission.


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path(".")
REPORTS = ROOT / "reports"
EMB = ROOT / "embeddings"

with (REPORTS / "part1_tfidf_top_words.json").open("r", encoding="utf-8") as f:
    tfidf_top = json.load(f)
with (REPORTS / "part1_w2v_eval.json").open("r", encoding="utf-8") as f:
    w2v_eval = json.load(f)
with (REPORTS / "part1_condition_comparison.json").open("r", encoding="utf-8") as f:
    condition_cmp = json.load(f)
with (REPORTS / "part2_results.json").open("r", encoding="utf-8") as f:
    part2 = json.load(f)
with (REPORTS / "part3_results.json").open("r", encoding="utf-8") as f:
    part3 = json.load(f)
comparison_text = (REPORTS / "part3_bilstm_vs_transformer.txt").read_text(encoding="utf-8")

print("Loaded report artifacts successfully.")


## Part 1 - Word Embeddings


In [ ]:
print("Top-10 TF-IDF discriminative words per topic")
print("=" * 72)
for topic, pairs in tfidf_top.items():
    print()
    print(f"[{topic}]")
    for rank, (word, score) in enumerate(pairs[:10], start=1):
        print(f"{rank:2d}. {word:15s} {score:.4f}")


In [ ]:
print("PPMI t-SNE plot (Top-200 frequent words)")
display(Image(filename=str(REPORTS / "part1_ppmi_tsne.png")))


In [ ]:
print("Top-5 nearest neighbours for 10 query words using PPMI matrix")
ppmi = np.load(EMB / "ppmi_matrix.npy")
with (EMB / "word2idx.json").open("r", encoding="utf-8") as f:
    word2idx = json.load(f)
idx2word = [None] * len(word2idx)
for w, i in word2idx.items():
    if i < len(idx2word):
        idx2word[i] = w

row_norm = np.linalg.norm(ppmi, axis=1, keepdims=True) + 1e-12
ppmi_norm = ppmi / row_norm

queries = ["pakistan", "hukumat", "adalat", "maeeshat", "fauj", "sehat", "taleem", "aabadi", "education", "hospital"]
for q in queries:
    if q not in word2idx:
        print()
        print(f"{q}: missing from vocabulary")
        continue
    qi = word2idx[q]
    sims = ppmi_norm @ ppmi_norm[qi]
    order = np.argsort(-sims)
    print()
    print(f"{q}:")
    count = 0
    for j in order:
        if j == qi:
            continue
        tok = idx2word[j] if j < len(idx2word) else "<NA>"
        if tok in ["<PAD>", "<UNK>", "<CLS>"]:
            continue
        print(f"  - {tok:15s} {float(sims[j]):.4f}")
        count += 1
        if count >= 5:
            break


In [ ]:
print("Skip-gram training loss curve")
display(Image(filename=str(REPORTS / "part1_skipgram_loss.png")))


In [ ]:
print("Top-10 nearest neighbours for 8 Word2Vec query words")
print("=" * 72)
for query, neighbors in w2v_eval["nearest_neighbors"].items():
    print()
    print(f"{query}:")
    for rank, item in enumerate(neighbors[:10], start=1):
        word, sim = item
        print(f"{rank:2d}. {word:15s} {sim:.4f}")


In [ ]:
print("Analogy results: top-3 candidates")
print("=" * 72)
for analogy, preds in w2v_eval["analogies"].items():
    print()
    print(f"{analogy}")
    if not preds:
        print("  No candidates available (missing query terms in vocabulary).")
        continue
    for rank, item in enumerate(preds[:3], start=1):
        word, sim = item
        print(f"  {rank}. {word:15s} {sim:.4f}")


In [ ]:
print("Four-condition comparison with MRR")
print("=" * 72)
for cid in ["C1", "C2", "C3", "C4"]:
    desc = condition_cmp[cid]["description"]
    mrr = condition_cmp.get("mrr", {}).get(cid, 0.0)
    print(f"{cid}: {desc:32s} MRR={mrr:.4f}")


## Part 2 - BiLSTM Sequence Labeling


In [ ]:
print("POS and NER class-label distributions")
print("=" * 72)
print("POS distribution:")
for k, v in part2["label_distribution"]["pos"].items():
    print(f"  {k:8s}: {v}")

print()
print("NER distribution:")
for k, v in part2["label_distribution"]["ner"].items():
    print(f"  {k:8s}: {v}")


In [ ]:
print("BiLSTM training and validation curves")
for name in [
    "part2_pos_frozen_curve.png",
    "part2_pos_finetuned_curve.png",
    "part2_ner_crf_curve.png",
    "part2_ner_softmax_curve.png",
]:
    print()
    print(f"{name}")
    display(Image(filename=str(REPORTS / name)))


In [ ]:
pos_cm = np.array(part2["pos"]["confusion_matrix"])
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(pos_cm, cmap="Blues")
ax.set_title("POS Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
print("NER conlleval-style entity scores (per type + overall)")
print("=" * 72)
for entity, vals in part2["ner"]["with_crf"].items():
    print(f"{entity:8s} P={vals['precision']:.4f} R={vals['recall']:.4f} F1={vals['f1']:.4f}")

print()
print("CRF vs No-CRF (overall F1)")
f1_crf = part2["ner"]["with_crf"]["overall"]["f1"]
f1_nocrf = part2["ner"]["without_crf"]["overall"]["f1"]
print(f"With CRF    : {f1_crf:.4f}")
print(f"Without CRF : {f1_nocrf:.4f}")


In [ ]:
print("Ablation study (A1-A4)")
print("=" * 72)
for aid, vals in part2["ablations"].items():
    desc = vals.get("description", "")
    pos_f1 = vals.get("pos_macro_f1", None)
    ner_f1 = vals.get("ner_overall_f1", None)
    print(f"{aid}: {desc}")
    if pos_f1 is not None:
        print(f"  POS macro-F1: {pos_f1:.4f}")
    if ner_f1 is not None:
        print(f"  NER overall-F1: {ner_f1:.4f}")


## Part 3 - Transformer Encoder Topic Classification


In [ ]:
print("Transformer training/validation curves")
display(Image(filename=str(REPORTS / "part3_transformer_curves.png")))

print()
print("BiLSTM baseline training/validation curves")
display(Image(filename=str(REPORTS / "part3_bilstm_curves.png")))


In [ ]:
cm = np.array(part3["transformer"]["confusion_matrix"])
labels = ["Politics", "Sports", "Economy", "International", "Health&Soc"]
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Purples")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
ax.set_title("Transformer Topic Confusion Matrix (5x5)")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black", fontsize=9)
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
print("Attention heatmaps (2 heads x 3 correctly classified articles)")
for name in [
    "part3_attention_article1_head0.png",
    "part3_attention_article1_head1.png",
    "part3_attention_article2_head0.png",
    "part3_attention_article2_head1.png",
    "part3_attention_article3_head0.png",
    "part3_attention_article3_head1.png",
]:
    print()
    print(f"{name}")
    display(Image(filename=str(REPORTS / name)))


In [ ]:
print("BiLSTM vs Transformer written comparison (10-15 sentences)")
print("=" * 72)
print(comparison_text)


## Conclusion

All required assignment outputs are included in this notebook as executed cell outputs.
The GitHub URL is included at the top of the notebook.
